# 프로젝트 - 항공사 AI 어시스턴트

지금까지 배운 것을 모아 **항공사 고객 지원 AI 어시스턴트**를 만듭니다.

In [1]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [2]:
# 초기화

load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API 키 있음, 접두사: {openai_api_key[:8]}")
else:
    print("OpenAI API 키 없음")
    
MODEL = "gpt-4.1-mini"
openai = OpenAI()

# OpenAI 대신 Ollama를 쓰려면:
# Ollama가 로컬에서 실행 중인지 확인하고(week1/day2 연습 참고) 아래 두 줄 주석 해제
# MODEL = "llama3.2"
# openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')


OpenAI API 키 있음, 접두사: sk-proj-


In [3]:
system_message = """
당신은 FlightAI 항공사의 고객 지원 어시스턴트입니다.
한 문장 이내로 짧고 정중하게 답하세요.
정확하게 답하고, 모르면 모른다고 하세요.
"""

In [4]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7868
* To create a public link, set `share=True` in `launch()`.


## 도구(Tools)

Tools는 Frontier LLM이 제공하는 매우 강력한 기능입니다.

함수를 하나 만들고, LLM이 **응답 과정에서 그 함수를 호출**하도록 할 수 있습니다.

우리 머신에서 코드를 실행할 수 있게 해 준다는 거라 조금 무섭게 들리지만, 그런 셈입니다.

In [5]:
# 먼저 쓸모 있는 함수 하나 만들기

ticket_prices = {"london": "$799", "paris": "$899", "tokyo": "$1400", "berlin": "$499"}

def get_ticket_price(destination_city):
    print(f"도구 호출: 도시 {destination_city}")
    price = ticket_prices.get(destination_city.lower(), "Unknown ticket price")
    return f"{destination_city} 행 티켓 가격은 {price}입니다"


In [6]:
get_ticket_price("London")

도구 호출: 도시 London


'London 행 티켓 가격은 $799입니다'

In [109]:
# 함수를 설명할 때 쓰는 특정 딕셔너리 구조가 필요합니다:

price_function = {
    "name": "get_ticket_price",
    "description": "목적지 도시까지 왕복 티켓 가격을 조회합니다.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "고객이 가고 싶어 하는 도시",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}
set_price_function = {
    "name": "set_ticket_price",
    "description": "목적지 도시까지 왕복 티켓 가격을 설정합니다.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "고객이 가고 싶어 하는 도시",
            },
            "destination_city_price": {
                "type": "number",
                "description": "가격",
            },
        },
        "required": ["destination_city", "destination_city_price"],
        "additionalProperties": False
    }
}

In [110]:
# 이걸 도구 목록에 넣습니다:

tools = [{"type": "function", "function": price_function},{"type": "function", "function": set_price_function}]

In [111]:
tools

[{'type': 'function',
  'function': {'name': 'get_ticket_price',
   'description': '목적지 도시까지 왕복 티켓 가격을 조회합니다.',
   'parameters': {'type': 'object',
    'properties': {'destination_city': {'type': 'string',
      'description': '고객이 가고 싶어 하는 도시'}},
    'required': ['destination_city'],
    'additionalProperties': False}}},
 {'type': 'function',
  'function': {'name': 'set_ticket_price',
   'description': '목적지 도시까지 왕복 티켓 가격을 설정합니다.',
   'parameters': {'type': 'object',
    'properties': {'destination_city': {'type': 'string',
      'description': '고객이 가고 싶어 하는 도시'},
     'destination_city_price': {'type': 'number', 'description': '가격'}},
    'required': ['destination_city', 'destination_city_price'],
    'additionalProperties': False}}}]

## OpenAI가 우리 도구를 쓰게 하기

OpenAI가 "우리 도구를 호출"하려면 처리할 게 조금 있습니다.

실제로 하는 일은, LLM이 **도구를 실행해 달라고 요청**할 수 있는 기회를 주는 것입니다.

새 chat 함수는 이렇게 됩니다:

In [112]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        response = handle_tool_call(message)
        messages.append(message)
        messages.append(response)
        response = openai.chat.completions.create(model=MODEL, messages=messages)
    
    return response.choices[0].message.content

In [113]:
# handle_tool_call 함수를 작성해야 합니다:

def handle_tool_call(message):
    tool_call = message.tool_calls[0]
    if tool_call.function.name == "get_ticket_price":
        arguments = json.loads(tool_call.function.arguments)
        city = arguments.get('destination_city')
        price_details = get_ticket_price(city)
        response = {
            "role": "tool",
            "content": price_details,
            "tool_call_id": tool_call.id
        }
    return response

In [114]:
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7891
* To create a public link, set `share=True` in `launch()`.


## 몇 가지 개선하기

한 번의 응답에서 여러 도구 호출 처리하기

연달아 여러 번 도구 호출 처리하기

In [115]:
def chat(message, history):
    # 1. [핵심 수정] 과거 기록(history)에 있는 모든 None 값을 빈 문자열("")로 청소합니다.
    history = [
        {"role": h["role"], "content": h["content"] if h["content"] is not None else ""} 
        for h in history
    ]
    
    # 2. 대화 맥락 생성
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    
    # 3. 모델에게 질문
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    if response.choices[0].finish_reason == "tool_calls":
        assistant_message = response.choices[0].message

        # 현재 메시지도 None 방어
        if assistant_message.content is None:
            assistant_message.content = ""
        
        # 4. 도구 실행
        tool_responses = handle_tool_calls(assistant_message)
        
        # 5. 메시지 추가
        messages.append(assistant_message)
        messages.extend(tool_responses)
        
        # 6. 최종 답변 생성 (여기서 에러가 났던 것이 해결됩니다)
        response = openai.chat.completions.create(model=MODEL, messages=messages)
    
    return response.choices[0].message.content

In [116]:
def handle_tool_calls(message):
    responses = []
    
    # 모델이 요청한 여러 개의 함수 호출(멀티 호출 가능)을 하나씩 처리
    for tool_call in message.tool_calls:
        # 호출하려는 함수 이름이 'get_ticket_price'인 경우
        if tool_call.function.name == "get_ticket_price":
            # 모델이 보낸 문자열 형태의 JSON 인자를 파이썬 딕셔너리로 변환
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            
            # [실제 비즈니스 로직] 데이터베이스나 API에서 가격 정보를 가져옴
            price_details = get_ticket_price(city)
            
            # OpenAI 규격에 맞춰 결과 메시지 구성
            responses.append({
                "role": "tool",                 # 역할은 반드시 'tool'이어야 함
                "content": price_details,       # 함수 실행 결과 (문자열)
                "tool_call_id": tool_call.id    # 모델이 보낸 ID와 일치시켜야 짝이 맞음
            })
        if tool_call.function.name == "set_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price = arguments.get('destination_city_price')
            price_details = set_ticket_price(city, price)
            # None이면 API/UI에서 오류 나므로 문자열로 보장
            responses.append({
                "role": "tool",
                "content": price_details if price_details is not None else "가격이 설정되었습니다.",
                "tool_call_id": tool_call.id
            })
    return responses

In [117]:
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7892
* To create a public link, set `share=True` in `launch()`.


DB 도구 호출: 한국 가격 조회 중
DB 도구 호출: 한국 가격 조회 중
DB 도구 호출: all 가격 조회 중
DB 도구 호출: 한국 가격 조회 중


In [75]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    return response.choices[0].message.content

In [76]:
import sqlite3


In [77]:
DB = "prices.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute('CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)')
    conn.commit()

In [78]:
def get_ticket_price(city):
    print(f"DB 도구 호출: {city} 가격 조회 중", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        return f"{city} 행 티켓 가격은 ${result[0]}입니다" if result else "해당 도시 가격 정보가 없습니다"

In [79]:
get_ticket_price("London")

DB 도구 호출: London 가격 조회 중


'London 행 티켓 가격은 $799.0입니다'

In [80]:
def set_ticket_price(city, price):
    if price is None:
        return "가격 값을 알려 주세요."
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?', (city.lower(), price, price))
        conn.commit()
    return f"{city} 행 티켓 가격이 ${price}로 설정되었습니다."

In [81]:
ticket_prices = {"london":799, "paris": 899, "tokyo": 1420, "sydney": 2999}
for city, price in ticket_prices.items():
    set_ticket_price(city, price)

In [82]:
get_ticket_price("Tokyo")

DB 도구 호출: Tokyo 가격 조회 중


'Tokyo 행 티켓 가격은 $1420.0입니다'

In [83]:
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7882
* To create a public link, set `share=True` in `launch()`.


DB 도구 호출: 한국 가격 조회 중
DB 도구 호출: 도쿄 가격 조회 중
DB 도구 호출: 도쿄 가격 조회 중
DB 도구 호출: 한국 가격 조회 중


## 연습

티켓 가격을 **설정**하는 도구를 추가해 보세요!

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">비즈니스 활용</h2>
            <span style="color:#181;">이제 LLM에게 **행동(액션)**을 부여할 수 있습니다. 이 항공사 어시스턴트는 질문에 답하는 것을 넘어, 예약 API와 연동해 실제 예약까지 할 수 있습니다!</span>
        </td>
    </tr>
</table>